# Transcribe and translate (OpenAI-compatible vLLM Whisper)

Workshop defaults: project `whisper-workshop`, deployment `whisper-large-v3`.

1. Set `INFERENCE_BASE_URL` to the HTTPS route (no trailing slash) from OpenShift AI → Deployments or `oc get inferenceservice`.
2. Set `BEARER_TOKEN` from Secret `whisper-large-v3-sa` key `token` (OpenShift console Workloads → Secrets), or `oc extract secret/whisper-large-v3-sa -n whisper-workshop --keys=token --to=-`. Strip accidental newlines.
3. Set `MODEL_NAME` to an `id` from `GET /v1/models` (this sample serves `whisper-large-v3`).

This notebook talks to **audio** endpoints, not `/v1/chat/completions`. The workbench cannot use your laptop microphone — use the bundled WAVs or upload a file.

In [ ]:
from pathlib import Path

import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- edit these ---
INFERENCE_BASE_URL = "https://REPLACE-with-your-route-host"
BEARER_TOKEN = "REPLACE-with-token-from-secret-whisper-large-v3-sa".strip()
MODEL_NAME = "whisper-large-v3"

HEADERS = {"Authorization": f"Bearer {BEARER_TOKEN}"}
SESSION = requests.Session()
SESSION.verify = False

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "extras" / "audio").exists():
    for candidate in Path.cwd().resolve().parents:
        if (candidate / "extras" / "audio").is_dir():
            REPO_ROOT = candidate
            break

EN_WAV = REPO_ROOT / "extras" / "audio" / "en-sample.wav"
ES_WAV = REPO_ROOT / "extras" / "audio" / "es-sample.wav"
print("repo", REPO_ROOT)
print("en", EN_WAV.exists(), EN_WAV)
print("es", ES_WAV.exists(), ES_WAV)


In [ ]:
r = SESSION.get(f"{INFERENCE_BASE_URL}/v1/models", headers=HEADERS, timeout=120)
print("status", r.status_code)
print(r.text[:4000])
if r.ok:
    payload = r.json()
    ids = [m.get("id") for m in payload.get("data", []) if m.get("id")]
    print("\nUse this exact string as MODEL_NAME if audio calls fail:")
    for i in ids:
        print("  ", repr(i))


## Transcribe English (`/v1/audio/transcriptions`)

Expect a transcript close to: *Welcome to OpenShift AI...*

In [ ]:
with EN_WAV.open("rb") as fh:
    files = {"file": (EN_WAV.name, fh, "audio/wav")}
    data = {"model": MODEL_NAME}
    tr = SESSION.post(
        f"{INFERENCE_BASE_URL}/v1/audio/transcriptions",
        headers=HEADERS,
        files=files,
        data=data,
        timeout=300,
    )
print("status", tr.status_code)
print(tr.text[:4000])
if tr.ok:
    print("\nTranscript:", tr.json().get("text"))


## Translate Spanish to English (`/v1/audio/translations`)

Native Whisper translation is **speech → English only**. Turbo checkpoints typically **cannot** serve this endpoint.

In [ ]:
with ES_WAV.open("rb") as fh:
    files = {"file": (ES_WAV.name, fh, "audio/wav")}
    data = {"model": MODEL_NAME}
    tl = SESSION.post(
        f"{INFERENCE_BASE_URL}/v1/audio/translations",
        headers=HEADERS,
        files=files,
        data=data,
        timeout=300,
    )
print("status", tl.status_code)
print(tl.text[:4000])
if tl.ok:
    print("\nEnglish translation:", tl.json().get("text"))


## Optional: stream tokens for one complete clip

This is **not** live microphone ASR. `stream=true` streams output tokens after the server has the whole file.

In [ ]:
with EN_WAV.open("rb") as fh:
    files = {"file": (EN_WAV.name, fh, "audio/wav")}
    data = {"model": MODEL_NAME, "stream": "true"}
    st = SESSION.post(
        f"{INFERENCE_BASE_URL}/v1/audio/transcriptions",
        headers=HEADERS,
        files=files,
        data=data,
        timeout=300,
        stream=True,
    )
print("status", st.status_code)
for line in st.iter_lines(decode_unicode=True):
    if line:
        print(line)


## Optional: upload your own clip

Record on the laptop (Voice Memos, `arecord`, etc.), upload into Jupyter, and set `UPLOAD` to that path.

In [ ]:
UPLOAD = Path("")  # e.g. Path("/opt/app-root/src/my-clip.wav")

if not UPLOAD or not UPLOAD.exists():
    print("Set UPLOAD to an existing audio file to run this cell.")
else:
    with UPLOAD.open("rb") as fh:
        files = {"file": (UPLOAD.name, fh, "application/octet-stream")}
        data = {"model": MODEL_NAME}
        up = SESSION.post(
            f"{INFERENCE_BASE_URL}/v1/audio/transcriptions",
            headers=HEADERS,
            files=files,
            data=data,
            timeout=300,
        )
    print("status", up.status_code)
    print(up.text[:4000])
